# Metric Validation Gate

Before publishing KPI outputs to SQL or Power BI,
all core metrics are validated against their business
definitions and independent calculations.

1. Load Data

2. Generate KPIs

3. Sales KPIs

4. Inventory KPIs

5. Supplier KPIs

6. Forecast KPIs

7. Warehouse KPIs

8. Regional KPIs

9. KPI Summary

10. Business Recommendations

## Load Data

In [1]:
import os
import sys
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0,PROJECT_ROOT)

In [4]:
import pandas as pd
feature_df = pd.read_csv(
    "../data/processed/inventory_clean.csv"
)
feature_df.head()

,date,sku_id,warehouse_id,supplier_id,region,units_sold,inventory_level,supplier_lead_time_days,reorder_point,order_quantity,...,day,quarter,lead_time_category,lead_time_demand,projected_inventory_after_lead_time,lead_time_reorder_risk,projected_stockout_risk,forecast_error,absolute_forecast_error,forecast_direction
0,2024-01-01,sku_1,wh_1,sup_8,west,10,592,14,379,0,...,1,1,slow,119.28,472.72,False,False,-1.48,1.48,underforecast
1,2024-01-02,sku_1,wh_1,sup_8,west,17,575,14,379,0,...,2,1,slow,260.82,314.18,True,False,1.63,1.63,overforecast
2,2024-01-03,sku_1,wh_1,sup_8,north,35,540,14,379,0,...,3,1,slow,554.68,-14.68,True,True,4.62,4.62,overforecast
3,2024-01-04,sku_1,wh_1,sup_8,south,24,516,14,379,0,...,4,1,slow,272.02,243.98,True,False,-4.57,4.57,underforecast
4,2024-01-05,sku_1,wh_1,sup_8,west,21,495,14,379,0,...,5,1,slow,261.80,233.20,True,False,-2.30,2.30,underforecast


## Generate Metrics

In [7]:
from src.metrics import generate_metrics
metrics = generate_metrics(feature_df)
metrics.keys()

dict_keys(['sales', 'inventory', 'supplier', 'forecast', 'warehouse', 'region'])

In [8]:
from src.metrics import generate_metrics

metrics = generate_metrics(feature_df)

sales = metrics["sales"]

sales

{'total_units_sold': np.int64(1829668),
 'total_revenue': np.float64(33419991.04),
 'total_cogs': np.float64(22333930.270000003),
 'gross_profit': np.float64(11086060.77),
 'gross_margin': np.float64(0.33171944171771983),
 'average_units_sold': np.float64(20.051156164383563),
 'max_units_sold': np.int64(48),
 'min_units_sold': np.int64(0)}

### KPI Interpretation

• Total Units Sold measures the overall sales volume.

• Average Units Sold represents the average quantity sold per observation.

• Maximum Units Sold identifies peak demand.

• Minimum Units Sold indicates low-demand products.

## Inventory KPI

Business Goal

Evaluate inventory health and stock availability.

In [9]:
metrics["inventory"]

{'average_inventory_value': np.float64(1439024.6332054795),
 'ending_inventory_value': np.float64(1404610.84),
 'average_inventory_units': np.float64(117880.57808219179),
 'max_inventory': np.int64(990),
 'min_inventory': np.int64(168),
 'inventory_turnover': np.float64(15.52018621130228),
 'days_inventory_outstanding': np.float64(23.517759067490807),
 'reorder_risk_rate': np.float64(0.05246027397260274),
 'projected_stockout_risk_rate': np.float64(0.046235616438356164),
 'source_stockout_rate': np.float64(0.0)}

Inventory KPIs help identify stock shortages and inventory efficiency.

In [10]:
metrics["supplier"]

{'average_lead_time': np.float64(7.984),
 'median_lead_time': np.float64(8.0),
 'lead_time_std': np.float64(3.9079292935533267),
 'lead_time_p90': np.float64(13.0),
 'max_lead_time': np.int64(14),
 'min_lead_time': np.int64(2)}

Forecast KPIs measure demand planning quality.

In [11]:
metrics["warehouse"]

,warehouse_id,total_units_sold,average_inventory_units,average_inventory_value,total_cogs,reorder_risk_rate,projected_stockout_risk_rate,inventory_turnover,days_inventory_outstanding
0,wh_1,366843,23302.810959,270307.277699,4259298.99,0.052384,0.047781,15.757249,23.163942
1,wh_2,366004,24574.482192,332599.660027,4989196.51,0.052274,0.041863,15.000606,24.332350
2,wh_3,366051,22777.953425,266168.934521,4283431.10,0.052822,0.055671,16.092904,22.680804
3,wh_4,365705,23869.515068,276995.728164,4267297.62,0.051836,0.054466,15.405644,23.692615
4,wh_5,365065,23355.816438,292953.032795,4534706.05,0.052986,0.031397,15.479294,23.579887


In [12]:
metrics["region"]

,region,total_units_sold,average_inventory_units,average_inventory_value,total_cogs,reorder_risk_rate,projected_stockout_risk_rate,inventory_turnover,days_inventory_outstanding
0,east,459869,29722.887671,361979.172603,5607634.64,0.050321,0.047671,15.491595,23.561164
1,north,459774,29632.383562,362051.962493,5621710.43,0.054515,0.045514,15.527358,23.506897
2,south,456883,29390.312329,357724.654603,5557209.41,0.050985,0.045083,15.534880,23.495515
3,west,453142,29134.994521,357268.843507,5547375.79,0.054447,0.046971,15.527175,23.507174


In [14]:
metrics["forecast"]

{'wape': np.float64(0.1187728648038879),
 'forecast_accuracy': np.float64(0.8812271351961121),
 'forecast_bias': np.float64(0.0015399132520216787),
 'overforecast_rate': np.float64(0.5006136986301369),
 'underforecast_rate': np.float64(0.49455342465753427),
 'accurate_rate': np.float64(0.004832876712328767)}